## 1. Import libraries and prepare the environment
We use:
- **NLTK** for tokenization, stopword removal and lemmatization.
- **TF-IDF** to convert text into numerical features.
- **Logistic Regression** to classify the user's intent.
- **Scikit-learn metrics** to evaluate the classifier.

In [ ]:
!pip -q install nltk seaborn ipywidgets

In [ ]:
import re
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

print("Libraries imported successfully.")

## 2. Develop a labelled corpus (at least 40 queries across 5 or more intent categories)

The corpus below contains **48 example queries** across six intent categories. Each query is labelled with its intended category. This is a small educational training dataset, not a clinically validated dataset.

In [ ]:
data = {
    "general_information": [
        "What does heart health mean?",
        "Explain what the heart does in the body",
        "What is cardiovascular health?",
        "Tell me about the human heart",
        "What is the meaning of heart disease?",
        "Why is the heart an important organ?",
        "What is the cardiovascular system?",
        "Give me basic information about heart health"
    ],
    "symptoms": [
        "What are common signs of heart problems?",
        "Why do I feel my heart beating fast?",
        "What symptoms can occur with heart disease?",
        "Can heart problems cause tiredness?",
        "What does an irregular heartbeat feel like?",
        "What are possible signs of poor circulation?",
        "Can shortness of breath be related to the heart?",
        "What are warning signs that I should discuss with a doctor?"
    ],
    "risk_factors": [
        "What increases the risk of heart disease?",
        "Does smoking affect heart health?",
        "Can high blood pressure harm the heart?",
        "How does diabetes affect cardiovascular health?",
        "Does family history increase heart disease risk?",
        "Can high cholesterol affect the heart?",
        "Is lack of exercise a heart disease risk?",
        "How does age affect the risk of heart disease?"
    ],
    "prevention": [
        "How can I help prevent heart disease?",
        "What habits support a healthy heart?",
        "How can I reduce my heart disease risk?",
        "What should I do to protect my heart?",
        "How often should adults exercise for heart health?",
        "Why are regular health checkups important?",
        "What lifestyle changes support cardiovascular health?",
        "How can I maintain good heart health?"
    ],
    "tests_and_checkups": [
        "What is an ECG test?",
        "Why might a doctor recommend an echocardiogram?",
        "What does a cholesterol test measure?",
        "What is a blood pressure check?",
        "How is heart health checked by a doctor?",
        "What is a stress test for the heart?",
        "What does an EKG show?",
        "Why are heart checkups performed?"
    ],
    "healthy_lifestyle": [
        "What foods are part of a heart healthy diet?",
        "How does walking help heart health?",
        "Is reducing salt good for the heart?",
        "How does sleep affect cardiovascular health?",
        "Can stress affect the heart?",
        "What are examples of heart healthy activities?",
        "Why should I avoid tobacco for heart health?",
        "How can I build healthier daily habits for my heart?"
    ]
}

rows = []
for intent, queries in data.items():
    for query in queries:
        rows.append({"query": query, "intent": intent})

corpus = pd.DataFrame(rows)
print("Total labelled queries:", len(corpus))
print("Number of intent categories:", corpus["intent"].nunique())
display(corpus.groupby("intent").size().rename("number_of_queries").to_frame())
display(corpus.head(10))

## 3. Create a hierarchical taxonomy

A taxonomy organizes the heart-health knowledge into major categories and subcategories. The following structure contains four major categories, each with at least two subcategories.

In [ ]:
taxonomy = {
    "Heart Health Basics": [
        "Heart function",
        "Cardiovascular system"
    ],
    "Health Conditions and Warning Signs": [
        "Possible symptoms",
        "Risk factors"
    ],
    "Prevention and Daily Care": [
        "Healthy eating and activity",
        "Lifestyle habits"
    ],
    "Medical Support": [
        "Screening and tests",
        "Professional and emergency care"
    ]
}

for major_category, subcategories in taxonomy.items():
    print(major_category)
    for subcategory in subcategories:
        print("   └──", subcategory)

## 4. Identify domain concepts and relationships

The knowledge model below defines more than eight heart-health concepts and at least ten relationships between them. These are simplified educational relationships, not a complete medical knowledge graph.

In [ ]:
domain_concepts = [
    "Heart",
    "Blood",
    "Blood vessels",
    "Blood pressure",
    "Cholesterol",
    "Physical activity",
    "Nutrition",
    "Smoking",
    "Diabetes",
    "Heart disease",
    "ECG",
    "Healthcare professional"
]

relationships = [
    ("Heart", "pumps", "Blood"),
    ("Blood", "flows through", "Blood vessels"),
    ("Blood pressure", "describes force of blood against", "Blood vessel walls"),
    ("High blood pressure", "can increase risk of", "Heart disease"),
    ("High cholesterol", "can contribute to", "Plaque in arteries"),
    ("Physical activity", "supports", "Cardiovascular health"),
    ("Nutrition", "influences", "Cardiovascular health"),
    ("Smoking", "increases risk of", "Heart disease"),
    ("Diabetes", "is a risk factor for", "Cardiovascular disease"),
    ("ECG", "records electrical activity of", "Heart"),
    ("Healthcare professional", "interprets", "ECG"),
    ("Healthcare professional", "advises on", "Heart health")
]

print("Domain concepts:")
for concept in domain_concepts:
    print("-", concept)

print("\\nRelationships:")
for subject, relation, obj in relationships:
    print(f"{subject} --{relation}--> {obj}")

print("\\nTotal concepts:", len(domain_concepts))
print("Total relationships:", len(relationships))

## 5. NLP preprocessing

The text preprocessing pipeline performs:
1. Lowercase conversion.
2. Tokenization.
3. Stopword removal.
4. Lemmatization.

We deliberately keep negation words such as **no**, **not** and **never**, because removing them could change the meaning of a health query.

In [ ]:
stop_words = set(stopwords.words("english")) - {"no", "not", "never"}
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # 1. Convert text to lowercase
    text = text.lower()

    # 2. Keep words and numbers; remove punctuation
    text = re.sub(r"[^a-z0-9\\s]", " ", text)

    # 3. Tokenize the sentence
    tokens = word_tokenize(text)

    # 4. Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # 5. Lemmatize tokens
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return " ".join(tokens)

corpus["processed_query"] = corpus["query"].apply(preprocess_text)

print("Original query:")
print(corpus.loc[0, "query"])
print("\\nAfter preprocessing:")
print(corpus.loc[0, "processed_query"])

## 6. TF-IDF feature extraction and intent classification

TF-IDF converts each query into numerical features based on word importance. Logistic Regression learns to associate those features with the six intent labels.

We use a stratified train-test split so that each intent category is represented in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    corpus["query"],
    corpus["intent"],
    test_size=0.25,
    random_state=42,
    stratify=corpus["intent"]
)

# A single pipeline applies TF-IDF and then trains Logistic Regression.
model = Pipeline([
    ("tfidf", TfidfVectorizer(preprocessor=preprocess_text, ngram_range=(1, 2))),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

model.fit(X_train, y_train)
print("Model training completed.")
print("Training examples:", len(X_train))
print("Testing examples:", len(X_test))

## 7. Evaluate the model

We evaluate the classifier using accuracy, precision, recall, F1-score and a confusion matrix. Since the dataset is small, these scores are only a classroom demonstration and should not be interpreted as real-world clinical performance.

In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

print(f"Accuracy : {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall   : {recall:.3f}")
print(f"F1-score : {f1:.3f}")

print("\\nDetailed classification report:")
print(classification_report(y_test, y_pred, zero_division=0))

labels = sorted(corpus["intent"].unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
fig, ax = plt.subplots(figsize=(11, 8))
disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
plt.title("Confusion Matrix – Heart Health Intent Classifier")
plt.tight_layout()
plt.show()

## 8. Develop a domain-specific knowledge base

The knowledge base contains educational information for every intent category. The assistant retrieves the response associated with the predicted intent.

The information is general and does not provide an individual diagnosis or treatment plan.

In [ ]:
knowledge_base = {
    "general_information": (
        "The heart is a muscular organ that pumps blood through the body. "
        "The cardiovascular system includes the heart and blood vessels. "
        "This system delivers oxygen and nutrients to tissues."
    ),
    "symptoms": (
        "Possible heart-related warning signs can include chest discomfort, "
        "shortness of breath, unusual fatigue, dizziness, or palpitations. "
        "These symptoms can have many causes. A healthcare professional should "
        "assess symptoms, especially if they are new, severe, or worsening."
    ),
    "risk_factors": (
        "Factors associated with cardiovascular risk include high blood pressure, "
        "high cholesterol, diabetes, tobacco use, physical inactivity, age, and "
        "family history. A healthcare professional can help assess personal risk."
    ),
    "prevention": (
        "Heart-health supporting habits include regular physical activity, "
        "balanced eating, avoiding tobacco, adequate sleep, and routine health "
        "checkups. Individual advice should be discussed with a healthcare professional."
    ),
    "tests_and_checkups": (
        "Healthcare professionals may use tests such as blood pressure measurement, "
        "blood tests for cholesterol, an ECG, an echocardiogram, or a stress test "
        "depending on the person's symptoms and clinical situation."
    ),
    "healthy_lifestyle": (
        "A heart-supportive lifestyle can include vegetables, fruits, whole grains, "
        "appropriate activity, limiting excess salt, avoiding tobacco, managing stress, "
        "and getting adequate sleep. Choose changes appropriate to your health needs."
    )
}

print("Knowledge base created for", len(knowledge_base), "intent categories.")
for category in knowledge_base:
    print("-", category)

## 9. Implement reasoning rules

The assistant applies rules before returning an answer:
1. **Emergency escalation:** urgent warning phrases trigger an emergency message immediately.
2. **Low-confidence handling:** uncertain model predictions are not answered as if reliable.
3. **Unsupported-query handling:** queries outside the assistant's heart-health scope are redirected.
4. **Empty-query handling:** blank input asks the user to enter a question.
5. **General safety rule:** the assistant does not diagnose or prescribe and recommends professional advice for personal medical concerns.

Emergency guidance is intentionally checked before machine-learning classification.

In [ ]:
# Emergency phrases are illustrative triggers, not a complete symptom checker.
EMERGENCY_PHRASES = [
    "severe chest pain",
    "crushing chest pain",
    "chest pain and shortness of breath",
    "cannot breathe",
    "can't breathe",
    "difficulty breathing",
    "fainted",
    "unconscious",
    "signs of stroke",
    "face drooping",
    "sudden weakness"
]

def is_emergency_query(query):
    q = query.lower()
    return any(phrase in q for phrase in EMERGENCY_PHRASES)

def is_heart_health_related(query):
    keywords = [
        "heart", "cardiac", "cardiovascular", "blood pressure",
        "cholesterol", "ecg", "ekg", "chest", "pulse", "smoking",
        "tobacco", "diabetes", "exercise", "diet", "salt",
        "blood vessel", "palpitation", "breath", "circulation"
    ]
    q = query.lower()
    return any(word in q for word in keywords)

def assistant_response(query, confidence_threshold=0.30):
    query = query.strip()

    # Rule 1: Empty query handling
    if not query:
        return "Please enter a heart-health question."

    # Rule 2: Emergency escalation comes before classification
    if is_emergency_query(query):
        return (
            "EMERGENCY ALERT: This may describe a medical emergency. "
            "Call your local emergency number now (in India, call 112), "
            "or go to the nearest emergency department. Do not wait for this "
            "assistant or rely on an online response. If possible, have someone "
            "stay with the person."
        )

    # Rule 3: Unsupported-query handling
    if not is_heart_health_related(query):
        return (
            "I am a Heart Health Information Assistant and can answer general "
            "questions about heart health, symptoms, risk factors, prevention, "
            "tests and healthy lifestyle. Please ask a question in one of these areas."
        )

    # Rule 4: Classify intent and check confidence
    probabilities = model.predict_proba([query])[0]
    best_index = int(np.argmax(probabilities))
    predicted_intent = model.classes_[best_index]
    confidence = float(probabilities[best_index])

    if confidence < confidence_threshold:
        return (
            "I am not sufficiently confident about the intent of that question. "
            "Please rephrase it or consult a healthcare professional for personal advice."
        )

    # Rule 5: Retrieve the relevant knowledge-base response
    answer = knowledge_base[predicted_intent]

    return (
        f"Detected intent: {predicted_intent.replace('_', ' ').title()}\\n"
        f"Model confidence: {confidence:.2%}\\n\\n"
        f"Assistant: {answer}\\n\\n"
        "Reminder: This is general educational information, not a diagnosis or treatment plan."
    )

print("Reasoning rules and response function are ready.")

## 10. Integrate NLP, classification, reasoning and knowledge retrieval

The following function combines preprocessing, intent classification, confidence checking, rule-based reasoning and knowledge retrieval into one assistant.

In [ ]:
def heart_health_assistant(user_query):
    print("User:", user_query)
    print("\\nAssistant:")
    print(assistant_response(user_query))

# Example
heart_health_assistant("How can walking help my heart?")

## 11. Test the assistant with the required queries

The assignment asks us to test:
1. A query belonging to each defined intent.
2. A differently worded query.
3. An unsupported healthcare query.
4. A low-confidence query.
5. An emergency-related query.

In [ ]:
# Test 1: One query for every defined intent
intent_test_queries = [
    "What does the heart do?",
    "What are common signs of heart problems?",
    "Does smoking increase heart disease risk?",
    "How can I help prevent heart disease?",
    "What is an ECG test?",
    "What foods support a healthy heart?"
]

for query in intent_test_queries:
    print("=" * 80)
    heart_health_assistant(query)

In [ ]:
# Test 2: Differently worded query
# This query expresses a familiar idea in a different way.
print("DIFFERENTLY WORDED QUERY")
heart_health_assistant("What daily habits can keep my ticker in good shape?")

In [ ]:
# Test 3: Unsupported query
print("UNSUPPORTED QUERY")
heart_health_assistant("How do I repair a bicycle tyre?")

In [ ]:
# Test 4: Low-confidence handling demonstration
# Use a deliberately high threshold to demonstrate the low-confidence rule.
print("LOW-CONFIDENCE HANDLING TEST")
print(assistant_response(
    "Tell me something about my health",
    confidence_threshold=0.99
))

In [ ]:
# Test 5: Emergency-related query
print("EMERGENCY QUERY")
heart_health_assistant("I have severe chest pain and difficulty breathing.")

## 12. Interactive interface – ask multiple questions

Run this cell to use the assistant interactively in Google Colab. Enter a question and click **Ask Assistant**. You can submit multiple queries one after another.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

question_box = widgets.Text(
    value="",
    placeholder="Type your heart-health question here...",
    description="Question:",
    layout=widgets.Layout(width="85%")
)

ask_button = widgets.Button(
    description="Ask Assistant",
    button_style="primary",
    icon="question"
)

output_area = widgets.Output()

def on_ask_clicked(button):
    with output_area:
        print("-" * 75)
        heart_health_assistant(question_box.value)
        print("-" * 75)

ask_button.on_click(on_ask_clicked)

display(widgets.VBox([
    widgets.HTML("<h3>Heart Health Information Assistant</h3>"),
    question_box,
    ask_button,
    output_area
]))

## Result

A Python-based Heart Health Information Assistant was developed in Google Colab. It uses a labelled corpus with 48 queries across six intent categories, a hierarchical taxonomy, domain concepts and relationships, NLP preprocessing, TF-IDF feature extraction and a Logistic Regression classifier.

The classifier is evaluated using accuracy, precision, recall, F1-score and a confusion matrix. A domain-specific knowledge base and rule-based reasoning handle normal questions, low-confidence queries, unsupported questions and emergency-related phrases. An interactive interface allows users to ask multiple questions.

## Conclusion

Thus, the Heart Health Information Assistant was implemented by integrating Natural Language Processing, machine-learning-based intent classification, rule-based reasoning and knowledge retrieval in Google Colab.

**Limitation:** This is a small academic prototype. Its training corpus, confidence score and emergency phrase list are not clinically validated. It must not be used to diagnose a condition, decide whether symptoms are safe, or delay emergency care.